In [ ]:
import numpy as np

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Methodology.EvaluationELM import EvaluationELM
from Pipeline.Global.GlobalSetting import GlobalSetting

In [ ]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train_scaled.shape[1]

In [ ]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x_train= gallstone_dataset.x_train,
    y_train= gallstone_dataset.y_train,
    activation_function     = GlobalSetting.sigmoid ,
    elm_init_seed_range= GlobalSetting.elm_initial_state_range ,
    k_fold = 5
)

In [ ]:
combination_record , combination_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = GlobalSetting.hidden_size_explore_range,
    lambda_range        = GlobalSetting.lambda_value_explore_range
)
GlobalSetting.save_dataframe_to_record(combination_record,'combination_record')
combination_result_list = EvaluationELM.extract_top_results(combination_result)
best_combination_result = combination_result_list.iloc[0]

In [ ]:
best_hidden_config = GlobalSetting.get_config_by_type("Best_Hidden_Nodes")
best_hidden_size = best_hidden_config["Hidden_Nodes"] if best_hidden_config else GlobalSetting.hidden_size_explore_range[len(GlobalSetting.hidden_size_explore_range)-1]

In [ ]:
hidden_size_smaller = range(11, round(best_hidden_size * 0.8))
lambda_value_wider = 1.44 ** np.arange(-50, 6)

In [ ]:
optimization_record , optimization_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_smaller,
    lambda_range=lambda_value_wider
)
GlobalSetting.save_dataframe_to_record(optimization_record,'optimization_record.csv')
optimization_result_list = EvaluationELM.extract_top_results(optimization_result)
best_optimization_result = optimization_result_list.iloc[0]

In [ ]:
new_configs_payload = [
    {
        "Model_Types" : "Grid_Combination",
        "Hidden_Nodes": int(best_combination_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_combination_result['Lambda_Value'])
    },
    {
        "Model_Types" : "Grid_Optimization",
        "Hidden_Nodes": int(best_optimization_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_optimization_result['Lambda_Value'])
    }
]
GlobalSetting.upsert_model_configs(new_configs_payload)

In [ ]:
best_optimization_result